# Week 6 · Random Variables, Sampling Distributions, and the Central Limit Theorem

*DS 207 · Week 6*

**Assignment Notebook**

**Author:** EC Corro, built in collaboration with Claude, ChatGPT, and Gemini

**Business scenario.** Visayas Fresh Logistics is a Cebu-based refrigerated delivery company serving supermarkets, hospitals, and restaurants under contracts that require at least 95% on-time delivery and box temperatures of 5°C or below. This assignment focuses on the provincial network, the North Cebu and South Cebu routes (3,000 completed deliveries in the ledger), where transit times are longer, more variable, and more strongly skewed than on the urban routes studied in the lecture notes. A late delivery carries a penalty of ₱0 if on time, ₱200 if up to 30 minutes late, and ₱500 if more than 30 minutes late.

**Submission.** Name your file `ds207_06_sampling_clt_lastname.ipynb`, replacing `lastname` with your surname, and upload it to your shared Google Drive folder by 11:59 PM on the day after the associated session, unless announced otherwise.

**AI use policy.** Generative AI is allowed for concept clarification, pseudocode, debugging, and idea generation. You must be able to explain any answer you submit without AI assistance, and you must cite any AI use.

**Points.** Section 1 (30 pts) + Section 2 (30 pts) = 60 pts.

## Learning Objectives

- Model delivery outcomes as discrete or continuous random variables and compute their expectation and variance.
- Derive and verify the sampling distribution of a sample mean, including its mean and standard error.
- Apply the Central Limit Theorem to a skewed variable, and judge when the normal approximation is and is not reliable.
- Turn probability calculations into a clear recommendation for an operational or contractual decision.

## Section 1: Concepts, Computation & Coding

This section is worth 30 points, 10 for each item. Run the cell below once before starting. It loads the provincial ledger into the DataFrame `deliveries`, with columns `route`, `delivery_min`, `promised_min`, `temp_c`, `late`, and `penalty_php`. Use `np.random.default_rng(207)` for every simulation so your results can be compared.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

import numpy as np
import pandas as pd

# Delivery ledger for Visayas Fresh Logistics: one row per completed cold-chain delivery.
rng = np.random.default_rng(207)

routes = {
    # route: (deliveries, median transit minutes, log-scale sigma, promised window, mean box temp, sd box temp)
    "Metro Cebu":  (3200, 26, 0.46, 75, 3.0, 0.8),
    "Mactan":      (1800, 30, 0.50, 90, 3.1, 0.8),
    "North Cebu":  (1600, 40, 0.58, 120, 3.4, 0.9),
    "South Cebu":  (1400, 38, 0.56, 120, 3.3, 0.9),
}

frames = []
for name, (n, med, sig, promised, t_mu, t_sd) in routes.items():
    minutes = 15 + rng.lognormal(mean=np.log(med), sigma=sig, size=n)
    frames.append(pd.DataFrame({
        "route": name,
        "delivery_min": minutes.round(1),
        "promised_min": promised,
        "temp_c": rng.normal(t_mu, t_sd, size=n).round(2),
    }))

ledger = pd.concat(frames, ignore_index=True)
ledger.insert(0, "delivery_id", np.arange(1, len(ledger) + 1))
delay = ledger["delivery_min"] - ledger["promised_min"]
ledger["late"] = (delay > 0).astype(int)
ledger["penalty_php"] = np.select([delay > 30, delay > 0], [500, 200], default=0)

# Assignment window: the provincial network (North Cebu and South Cebu routes)
deliveries = ledger[ledger["route"].isin(["North Cebu", "South Cebu"])].reset_index(drop=True)
deliveries.head()

,delivery_id,route,delivery_min,promised_min,temp_c,late,penalty_php
0,5001,North Cebu,56.0,120,2.49,0,0
1,5002,North Cebu,51.0,120,3.66,0,0
2,5003,North Cebu,148.6,120,2.19,1,200
3,5004,North Cebu,48.1,120,1.96,0,0
4,5005,North Cebu,26.0,120,3.40,0,0


### Item 1 · Penalties as a random variable (10 pts)

On the provincial routes, the penalty $X$ (in pesos) charged on a delivery is a discrete random variable. Rounded to three decimals, the ledger gives:

| Penalty $x$ | ₱0 | ₱200 | ₱500 |
|---|---|---|---|
| $p(x)$ | 0.957 | 0.027 | 0.016 |

Each week a provincial hub audits 25 deliveries and counts how many were late.

**(a) Concept (3 pts).** Classify transit minutes and the weekly late count as discrete or continuous. State the conditions under which the late count is a binomial random variable, and describe one realistic way the independence condition could fail on the provincial routes.

**(b) By hand (4 pts).** Using the table above, compute $E[X]$, $\operatorname{Var}(X)$, and $\operatorname{SD}(X)$. Then, treating the 25 audited deliveries as independent, find the expected value and standard deviation of the total penalty $T$ for one audit. Show your work.

**(c) Coding (3 pts).** From the full provincial ledger, compute the empirical probability table for `penalty_php` and its exact mean and variance, and compare them with your answers to (b). Then simulate 10,000 audits of 25 deliveries, and report the mean and standard deviation of the audit total.

**Answer (a)**
Transit minutes are a **continuous random variable** while weekly late count is a **discrete random variable**. Discrete random variables are values that can be counted while continuous random variables are values that fill an interval with infinitely many possibilities.    

Binomial random variables are numbers or values of "successes" in a fixed number of independent trials. By definition, if X, a random variable, represents the number of successes that occur in the n trials, then X is said to be a binomial random variable with parameters (n,p), where p is the probability that the trial is a success. 

With this definition, we can say that the conditions under which the late count is a binomial random variable if a delivery is either delivered late or delivered on time given a fixed number of audited deliveries. Additionally, a binomial condition satisfies if the probability of lateness p is constant accross deliveries and if deliveries are independent of oone another.  

One way that the independence condition could fail is maybe if there's a weather disruption and caused multiple deliveries to be late in the same week. 

**Answer (b)**
| Penalty $x$ | ₱0 | ₱200 | ₱500 |
|---|---|---|---|
| $p(x)$ | 0.957 | 0.027 | 0.016 |

$$E[X] = 0(0.957) + 200(0.027) + 500(0.016) = 5.4 + 8 = 13.5 \text{ pesos}$$
$$E[X^2] = 200^2(0.027) + 500^2(0.016) = 1080 + 4000 = 5080$$
$$\operatorname{Var}(X) = 5080 - (13.5)^2 = 4897.75, \qquad \operatorname{SD}(X) = \sqrt{4897.75} = 69.98 \text{ pesos}$$

Treating the 25 deliveries as independent, the total penalty $T$ has

$$E[T] = 25 \times 13.5 = 337.5, \qquad \operatorname{Var}(T) = 25 \times 4897.75 = 122443.75, \qquad \operatorname{SD}(T) = 349.92.$$

In [2]:
# TODO
# Frequency table
penalty_counts = deliveries["penalty_php"].value_counts(normalize=True).sort_index()
penalty_table = penalty_counts.round(4)

# Mean and variance
mean_penalty = deliveries["penalty_php"].mean()
var_penalty = deliveries["penalty_php"].var()

print("Empirical probability table:")
print(penalty_table)
print(f"Mean penalty: {mean_penalty:.2f}")
print(f"Variance penalty: {var_penalty:.2f}")

# Simulation: 10,000 audits of 25 deliveries each
rng = np.random.default_rng(207)
n_audits = 10000
audit_totals = []

for _ in range(n_audits):
    sample = deliveries["penalty_php"].sample(25, replace=True, random_state=rng)
    audit_totals.append(sample.sum())

audit_totals = np.array(audit_totals)

print(f"Mean of audit totals: {audit_totals.mean():.2f}")
print(f"SD of audit totals: {audit_totals.std():.2f}")


Empirical probability table:
penalty_php
0      0.9570
200    0.0267
500    0.0163
Name: proportion, dtype: float64
Mean penalty: 13.50
Variance penalty: 4969.41
Mean of audit totals: 341.87
SD of audit totals: 354.08


### Item 2 · The sampling distribution of an audit average (10 pts)

A small feeder route has only three regular deliveries, with transit times of 40, 55, and 100 minutes. An auditor draws two deliveries at random with replacement and records their average transit time $\bar{X}$.

**(a) Concept (3 pts).** Explain the difference between the distribution of a single delivery time and the sampling distribution of the audit average. Say what stays the same between them, what changes, and why.

**(b) By hand (4 pts).** List all nine equally likely ordered samples and their means, and build the sampling distribution table of $\bar{X}$. Compute $E[\bar{X}]$ and $\operatorname{Var}(\bar{X})$ directly from the table, and verify them against $\mu$ and $\sigma^2/n$ for the three-delivery population.

**(c) Coding (3 pts).** Using `delivery_min` from the full provincial ledger, compute the population mean $\mu$ and population standard deviation $\sigma$ (use `ddof=0`). Simulate 10,000 audits of $n = 30$ deliveries, report the mean and standard deviation of the audit averages, compare them with $\mu$ and $\sigma/\sqrt{30}$, and report whether the audit averages are less skewed than individual delivery times.

**Answer (a)**
A sample distribution is the distribution of data given a random sample size drawn from the population. The population is composed of the delivery time. 

**Answer (b)**
The mean and variance of the three single deliveries, with transit times of 40, 55, and 100 minutes are the following:
$$\mu = \frac{40 + 55 + 100}{3} = 65, \qquad \sigma^2 = \frac{(40-65)^2 + (55-65)^2 + (100-65)^2}{3} = \frac{625 + 100 + 1225}{3} = 650.$$

Draw samples of size $n = 2$ with replacement. There are $3 \times 3 = 9$ equally likely ordered samples, and each has a sample mean. Listing all 9 gives the sampling distribution of $\bar{X}$:

| Sample mean $\bar{x}$ | 40 | 47.5 | 55 | 70 | 77.5 | 100 | 
|---|---|---|---|---|---|---|
| Number of samples | 1 | 2 | 1 | 2 | 2 | 1 | 
| Probability | 1/9 | 2/9 | 1/9 | 2/9 | 2/9 | 1/9 | 

$$E[\bar{X}] = \frac{40 + 2(47.5) + 55 + 2(70) + 2(77.5) + 100 }{9} = \frac{585}{9} = 65 = \mu.$$
$$\operatorname{Var}(\bar{X}) = \frac{625 + 2(306.25) + 100 + 2(25) + 2(156.25) + 1225}{9} = \frac{2925}{9} = 325 = \frac{\sigma^2}{n} = \frac{650}{2}.$$

In [ ]:
# TODO

### Item 3 · The Central Limit Theorem and audit size (10 pts)

A provincial hub audits $n = 36$ deliveries each week. For this network, single delivery times have mean $\mu = 61.2$ minutes, standard deviation $\sigma = 29.4$ minutes, and skewness of about 2.3.

**(a) Concept (3 pts).** Explain why the Central Limit Theorem allows a normal-distribution calculation for the weekly audit average even though single delivery times are strongly skewed. State one thing the theorem does not guarantee at $n = 36$, and explain how the skewness of the data affects the sample size needed.

**(b) By hand (4 pts).** Using the normal approximation and a standard normal table, compute (i) the standard error of the audit average, (ii) $P(\bar{X} > 70)$, (iii) the interval centered on $\mu$ that contains 95% of audit averages, and (iv) the smallest audit size $n$ for which the 95% margin of error for the mean is at most 5 minutes.

**(c) Coding (3 pts).** Simulate 20,000 audits of $n = 36$ deliveries from the provincial ledger. Compare the simulated $P(\bar{X} > 70)$ with your normal answer, report the share of simulated audit averages inside your interval from (b)(iii), and report the skewness of the audit averages. Then simulate 10,000 audits at your sample size from (b)(iv) and report the share of audit averages within 5 minutes of $\mu$.

**Answer (a)**

**Answer (b)**
Given that we have n = 36, mu = 61.2 minutes, and a sigma = 29.4, skewness = 2.3:
(i)
$$\operatorname{SE} = \frac{29.4}{\sqrt{36}} = \frac{29.4}{6} = 4.9$$

(ii)
$$z = \frac{70 - 61.2}{4.9} = 1.8, \qquad P(\bar{X} > 70) \approx 1 - \Phi(1.8) = 1 - 0.9641 = 0.0359.$$

(iii)
$$1.96 \times 4.9 = 9.604 \text{ minutes.}$$

(iv)
The margin of error for a 95% confidence interval is 
$$\bar{x} \pm 1.96\,\frac{s}{\sqrt{n}}.$$
where z = 1.96 for 95% confidence. 

Since we already know that sigma = 29.4, we plug that in the formula above and we get:
$$\bar{x} \pm 1.96\,\frac{29.4}{\sqrt{n}} \leq 5.$$
Which gives us, $$n \geq (11.53)^2 \approx 132.9$$

Hence, the smallest audit size $n$ for which the 95% margin of error for the mean is at most 5 minutes is 133 deliveries. 

In [ ]:
# TODO

## Section 2: Business Context & Reasoning (30 pts)

Each item below presents a decision facing a stakeholder at Visayas Fresh Logistics. Use the provincial ledger statistics from Section 1, state your assumptions, support your answer with a calculation, and end with a recommendation. Both technical soundness and business reasoning are graded, so a correct calculation without a usable recommendation, or a sensible recommendation without a correct calculation, will not earn full marks.

### Item 4 · An alarming audit at the provincial hub (15 pts)

This week's audit at the Provincial Hub sampled 36 deliveries and found 5 late (13.9%). The Provincial Hub Manager must decide whether to escalate to the Operations Director and pull trucks off the route for inspection. The ledger shows a long-run late rate of 4.3% on provincial routes, and the client contracts allow at most 5%.

Write your response in the cells below. It should:

1. State the assumptions you make about how the audit was drawn and which late rate you use as the benchmark.
2. Choose and justify a probability model, compute the probability of seeing 5 or more late deliveries in an audit of 36 if the true late rate is 4.3%, and comment on whether a normal approximation would be reliable here.
3. Interpret the result and recommend what the Hub Manager should do, including any follow-up action.

**Point split:** technical soundness (8 pts): assumptions 2, model and computation 3, interpretation 3; business reasoning (7 pts): recommendation follows from the result 4, practical follow-up and risk awareness 3.

**Assumptions and technical analysis**
(1) We assume that the sample 36 deliveries are random and independetly drawn from the provincial ledger. The benchmark late rate is the 4.3% long-run provincial rate since that reflects the underlying process. The client contract allows at most 5%, so that's the threshold for escalation. 

(2) From Item 1.a, we have already established that late count is a binomial random variable. And if we base it on the conditions that we have for a binomial random variable, late counts satisfy all of these conditions. 

$$P(X \geq 5) = 1 - P(X \leq 4)$$
$$P(K = k) = \binom{36}{k} 0.0043^{k} (1-0.0043)^{36-k}$$
$$P(X \leq 4) = \sum_{i=1}^{4} \binom{36}{k} 0.0043^{k} (0.957)^{36-k}$$

$$P(X = 0) = \binom{36}{0} (0.0043)^{0} (0.957)^{36} \approx 0.21$$
$$P(X = 1) = \binom{36}{1} (0.0043)^{1} (0.957)^{36-1} \approx 0.338$$
$$P(X = 2) = \binom{36}{2} (0.0043)^{2} (0.957)^{36-2} \approx 0.263$$
$$P(X = 3) = \binom{36}{3} (0.0043)^{3} (0.957)^{36-3} \approx 0.127$$
$$P(X = 4) = \binom{36}{4} (0.0043)^{4} (0.957)^{36-4} \approx 0.043$$

$$P(X \leq 4) \approx 0.981$$
Thus, 
$$P(X \geq 5) = 1 - 0.981 = 0.019$$
So the probability is about 1.9%.

The chance of seeing 5 or more late deliveries under the 4.3% benchmark is only about 1.9 or 2 in 100 audits. 


In [3]:
# TODO


**Interpretation and recommendation**

### Item 5 · A proposed contract clause (15 pts)

A supermarket chain proposes this clause for provincial deliveries: if the average transit time in the weekly audit of 36 deliveries exceeds 65 minutes, Visayas Fresh Logistics pays a ₱20,000 penalty for that week. The Commercial Director asks whether to accept. Use $\mu = 61.2$ minutes and $\sigma = 29.4$ minutes for provincial single-delivery times.

Write your response in the cells below. It should:

1. State the assumptions you make, including why a normal calculation is or is not defensible here.
2. Compute the probability that a given week triggers the penalty by chance alone, the expected number of triggered weeks and expected penalty cost over 52 weeks, and the same figures if the audit were 100 deliveries.
3. Recommend whether to accept, reject, or counter-propose, with specific terms and the reasoning behind them.

**Point split:** technical soundness (8 pts): assumptions 2, computation 3, interpretation 3; business reasoning (7 pts): recommendation follows from the result 4, negotiation practicality and risk awareness 3.

**Assumptions and technical analysis**
1. Given n = 36 deliveries, mu = 61.2, sigma = 29.4, sd = 4.9. We assume that the delivery times are independent, and all provincial deliveries follow the same distribution. By Central Limit Theorem, the sample mean is approximately normal for n = 36, so a normal calculation is defensible here. 

2. 

In [ ]:
# TODO

**Interpretation and recommendation**

**Citation reminder.** All external sources, including textbooks, papers, articles, code snippets, and any use of AI tools, must be cited in accordance with course policy. Answers that rely on uncited external material will be marked down.

## References

Blitzstein, J. K., & Hwang, J. (2019). *Introduction to probability* (2nd ed.). CRC Press.

Casella, G., & Berger, R. L. (2002). *Statistical inference* (2nd ed.). Duxbury.

Ross, S. M. (2019). *A first course in probability* (10th ed.). Pearson.

Wasserman, L. (2004). *All of statistics: A concise course in statistical inference*. Springer.